<div style="background: #86d1f1ff; border-radius: 5px; padding: 1rem; margin-bottom: 1rem">
<img src="https://store.utec.edu.pe/files/Recursos/logo-utec-h.png" alt="Banner" width="150" />   
<div style="font-weight: bold; color: #434549ff; float: right "><u style="font-size: 28px;">Base de Datos II</u> <br />
<span style="float:right"> Profesor Heider Sanchez</span> <br /> 
<span style="float:right">  2026 - 1 </span>   
</div> </div>

# Laboratorio 8.2: Similitud de Coseno e Indice Invertido

> **Prof. Heider Sanchez**  

> Grupo: Juan Ticlia, Paulo Miranda, Jose Huaman

## Introducción

Este laboratorio extiende las capacidades desarrolladas en el laboratorio 7.1, enfocándose en técnicas avanzadas de recuperación de información. Utilizaremos los Bag of Words previamente generados para implementar dos funcionalidades esenciales en los motores de búsqueda modernos: el **Índice Invertido** para recuperación eficiente de documentos, y la **Similitud de Coseno** para resultados ordenados por relevancia.


### Objetivos
- Implementar la conexión a PostgreSQL para extraer id, contenido y bag_of_words de los documentos.
- Construir el índice invertido (posting lists), calcular estadísticas IDF y la norma vectorial de cada documento.
- Implementar búsquedas booleanas (AND, OR y AND-NOT) con complejidad O(n+m) usando el índice invertido.
- Implementar búsquedas rankeadas usando la Similitud de Coseno:
  - Procesar consultas en lenguaje natural aplicando tokenización y cálculo del TF.
  - Utilizar el índice invertido para obtener los documentos que intersectan con la query.
  - Calcular similitud de coseno con TF-IDF entre la query y los documentos recuperados.
  - Devolver los top-k resultados ordenados por relevancia.
  - **Evitar usar la representación vectorial dispersa.**

In [29]:
import psycopg2
import pandas as pd

def connect_db():
    conn = psycopg2.connect(
        dbname="postgres",
        user="postgres",
        password="123456",
        host="localhost",
        port="5432",
    )
    return conn

def fetch_data():
    conn = connect_db()
    query = "SELECT id, contenido, bag_of_words FROM noticias;"
    df = pd.read_sql(query, conn)
    # El atributo está como jsonb en la db, cuadno llega a python no es necesario castearlo (da error)
    # por eso estoy comentando la linea que sigue:
    # df['bag_of_words'] = df['bag_of_words'].apply(json.loads)
    conn.close()
    return df

noticias_df = fetch_data()
print(noticias_df)

        id                                          contenido  \
0        1  Durante el foro La banca articulador empresari...   
1        2  El regulador de valores de China dijo el domin...   
2        3  En una industria históricamente masculina como...   
3        4  Con el dato de marzo el IPC interanual encaden...   
4        5  Ayer en Cartagena se dio inicio a la versión n...   
...    ...                                                ...   
1212  1211  La XVI Cumbre de la Alianza del Pacífico se ll...   
1213  1212  La inflación en España volvió a dispararse en ...   
1214  1214  La espiral alcista de los precios continúa y g...   
1215  1215  Las grandes derrotas nacionales son experienci...   
1216  1216  BBVA ha alcanzado un acuerdo de colaboración c...   

                                           bag_of_words  
0     {'aca': 1, 'adn': 1, 'cas': 1, 'for': 1, 'ret'...  
1     {'dat': 1, 'deb': 1, 'inc': 1, 'med': 1, 'mes'...  
2     {'go': 1, 'us': 3, 'air': 1, 'ane': 1, 

/tmp/ipykernel_15184/2263868634.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


## 1. (6 puntos) Construcción del Indice Invertido 

A partir de los  `bag of words` almacenados en la base de datos  (Laboratorio 8.1), se debe construir un índice invertido y conservarlo en un diccionario de Python para su eficiente recuperación.

In [30]:

# Preproceso de un texto de lab 8.1
import nltk
from nltk.stem import SnowballStemmer
from nltk.tokenize import word_tokenize
nltk.download('punkt')
nltk.download('punkt_tab')
def fetch_stopwords():
    conn = connect_db()
    query = "SELECT word FROM stopwords;"
    df = pd.read_sql(query, conn)
    conn.close()
    stopwords_set = set(df["word"].str.lower())
    return stopwords_set
stemmer = SnowballStemmer("spanish")
stopwords = fetch_stopwords()
def preprocess(text):
    text = text.lower()
    text = "".join(c for c in text if c.isalpha() or c.isspace())
    tokens = word_tokenize(text, language='spanish')
    words = [t for t in tokens if t not in stopwords]
    words = [stemmer.stem(w) for w in words]
    return words

[nltk_data] Downloading package punkt to /home/tkaos/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/tkaos/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
/tmp/ipykernel_15184/64490081.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [31]:
import math

class InvertedIndex:
    def __init__(self):
        self.index = {}
        self.idf = {}
        self.length = {}
        N = 0

    def build_from_db(self):
        # Leer desde PostgreSQL todos los bag of words
        # Construir el índice invertido, el idf y la norma (longitud) de cada documento
        
        """
        index  = {
            "word1": [("doc1", tf1), ("doc2", tf2), ("doc3", tf3)],
            "word2": [("doc2", tf2), ("doc4", tf4)],
            "word3": [("doc3", tf3), ("doc5", tf5)],
        } 
        idf  = {
            "word1": 3,
            "word2": 2,
            "word3": 2,
        } 
        length = {
            "doc1": 15.5236,
            "doc2": 10.5236,
            "doc3": 5.5236,
        }
        """

        noticias_df = fetch_data()
        noticias_df = noticias_df.head(5)


        # INDEX: Construccion del indice invertido index = [(id, tf), ...]

        # Aclaracion. tf no normalizado
        
        for row in noticias_df.itertuples():
            doc_id = row.id
            bow = row.bag_of_words

            # Guardar en indice invertido
            for word in bow:
                tupla = (doc_id, bow[word])
                if word not in self.index:
                    self.index[word] = []
                self.index[word].append(tupla)

        # Ordenar cada lista de documentos correspondiente a cada palabra
        # para que al hacer las querys sea eficiente por merge
        for word in self.index:
            self.index[word].sort(key=lambda x: x[0])

        

        # IDF: Construccion del ranked retrieval idf

        # Aclaracion. Se usa log natural (base e)

        N = len(noticias_df)

        for word in self.index:
            df = len(self.index[word])
            self.idf[word] = math.log(N / df)

        

        # LENGTH: Construccion de la norma weight tf.idf de cada documento
        # Aclaracion. TF amortizado | tf = log_natural(1 + tf)

        for row in noticias_df.itertuples():
            doc_id = row.id
            bow = row.bag_of_words

            length = 0

            for word, tf in bow.items():
                tf = math.log(1 + tf)
                tfidf = tf * self.idf[word]
                length += tfidf ** 2

            self.length[doc_id] = math.sqrt(length)

    def L(self, word):
        # Retorna la lista de documentos que contienen a word
        tokens = preprocess(word)
        if not tokens:
            return []
        return self.index.get(tokens[0], [])

    def showDocuments(self, result):
        # Extraemos solo el doc_id
        doc_ids = [doc_id for doc_id, tf in result]
        return doc_ids

    def cosine_search(self, query, top_k=5):  
        score = {}
        # No es necesario usar vectores numericos del tamaño del vocabulario
        # Guiarse del algoritmo visto en clase
        # Se debe calcular el tf-idf de la query y de cada documento
        # La query se debe procesar  al igual como se procesaron los documentos (bag of words)
        
        # TODO
        
        # Ordenar el score resultante de forma descendente
        result = sorted(score.items(), key= lambda tup: tup[1], reverse=True)
        # retornamos los k documentos mas relevantes (de mayor similitud a la query)
        return result[:top_k] 
    


invi = InvertedIndex()
invi.build_from_db()
print(invi.index)
print(invi.idf)
print(invi.length)

/tmp/ipykernel_15184/2263868634.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


{'aca': [(1, 1)], 'adn': [(1, 1)], 'cas': [(1, 1)], 'for': [(1, 1), (5, 4)], 'ret': [(1, 2), (5, 2)], 'tem': [(1, 2)], 'abre': [(1, 1)], 'banc': [(1, 1), (2, 1), (5, 5)], 'bbva': [(1, 1), (5, 1)], 'cost': [(1, 1), (3, 4), (4, 1)], 'garc': [(1, 2)], 'jueg': [(1, 1)], 'logr': [(1, 1)], 'mund': [(1, 2), (2, 1)], 'real': [(1, 1)], 'trat': [(1, 1), (3, 1)], 'asoci': [(1, 2)], 'aspir': [(1, 1)], 'cambi': [(1, 2)], 'conoc': [(1, 1)], 'enorm': [(1, 1)], 'fisic': [(1, 2)], 'fuent': [(1, 1)], 'inclu': [(1, 1)], 'mayor': [(1, 1)], 'mejor': [(1, 1)], 'riesg': [(1, 1), (2, 1)], 'ahiest': [(1, 1)], 'andres': [(1, 1)], 'aspect': [(1, 1)], 'client': [(1, 1)], 'climat': [(1, 1)], 'compit': [(1, 1)], 'conclu': [(1, 1)], 'corpor': [(1, 1)], 'direct': [(1, 1), (5, 2)], 'entend': [(1, 1)], 'global': [(1, 1), (2, 1)], 'impact': [(1, 1)], 'import': [(1, 2), (5, 1)], 'impuls': [(1, 1), (4, 1)], 'manten': [(1, 1), (2, 1), (4, 1)], 'negoci': [(1, 5)], 'social': [(1, 2)], 'sosten': [(1, 3), (5, 1)], 'viabil': [(

## 2. (6 puntos) Consultas Booleanas usando el indice invertido

Implementar búsquedas booleanas utilizando el índice invertido construido anteriormente. La búsqueda debe:

- Soportar los operadores básicos:
    - AND: intersección de documentos
    - OR: unión de documentos
    - AND-NOT: diferencia de documentos
- Procesar consultas como:
    - "sostenibilidad AND ambiente AND renovable"
    - "tecnología AND (banca OR finanzas)"
    - "economía AND-NOT inflación"    

####  Pruebas funcionales

In [ ]:
idx = InvertedIndex()
idx.build_from_db()

def AND(list1, list2):
    # Implementar la intersección de dos listas O(n +m)
    # Si alguna de las listas está vacía, no hay intersección posible
    if not list1 or not list2:
        return []
    
    n, m = len(list1), len(list2)
    p1, p2 = 0, 0
    answer = []
    while p1 < n and p2 < m:
        doc_id1, tf1 = list1[p1]
        doc_id2, tf2 = list2[p2]
        if doc_id1 == doc_id2:
            answer.append((doc_id1, tf1 + tf2))
            p1 += 1
            p2 += 1
        elif doc_id1 < doc_id2:
            p1 += 1
        else:
            p2 += 1
    return answer

def OR(list1, list2):
    # Implementar la unión de dos listas O(n +m)
    # listas vacias
    if not list1: return list2
    if not list2: return list1
    
    n, m = len(list1), len(list2)
    p1, p2 = 0, 0
    answer = []
    while p1 < n and p2 < m:
        doc_id1, tf1 = list1[p1]
        doc_id2, tf2 = list2[p2]
        if doc_id1 == doc_id2:
            answer.append((doc_id1, tf1 + tf2))
            p1 += 1
            p2 += 1
        elif doc_id1 < doc_id2:
            answer.append(list1[p1])
            p1 += 1
        else:
            answer.append(list2[p2]) 
            p2 += 1
            
    # Vaciar los elementos restantes
    while p1 < n:
        answer.append(list1[p1])
        p1 += 1
    while p2 < m:
        answer.append(list2[p2])
        p2 += 1
        
    return answer

def AND_NOT(list1, list2):
    # Implementar la diferencia de dos listas O(n +m)
    n, m = len(list1), len(list2)
    p1, p2 = 0, 0
    answer = []
    while p1 < n and p2 < m:
        doc_id1, tf1 = list1[p1]
        doc_id2, tf2 = list2[p2]
        if doc_id1 == doc_id2:
            # Si está en ambos, se excluye
            p1 += 1
            p2 += 1
        elif doc_id1 < doc_id2:
            answer.append(list1[p1])
            p1 += 1
        else:
            p2 += 1
            
    # Si list2 se acabó, todo lo que quede en list1 NO está en list2
    while p1 < n:
        answer.append(list1[p1])
        p1 += 1
        
    return answer

# Prueba 1
result = AND(idx.L("sostenibilidad"), AND(idx.L("ambiente"), idx.L("renovables")))
print("sostenibilidad AND ambiente AND renovable: ", idx.showDocuments(result))

# Prueba 2
result = AND(idx.L("tecnología"), OR(idx.L("banca"), idx.L("finanzas")))
print("tecnología AND (banca OR finanzas): ", idx.showDocuments(result))

# Prueba 3
result = AND_NOT(idx.L("economía"), idx.L("inflación"))
print("economía AND-NOT inflación: " , idx.showDocuments(result))

#Agregar dos pruebas mas combinando los operadores AND, OR, AND_NOT
# Prueba 4
result = AND_NOT(AND(OR(idx.L("renovable"), idx.L("eléctrico")), idx.L("energía")), idx.L("contaminación"))
print("(renovable OR eléctrico) AND energía AND-NOT contaminación: ", idx.showDocuments(result))
# Prueba 5
result = AND_NOT(OR(idx.L("economía"), idx.L("mercado")),AND(idx.L("financiero"), idx.L("pérdidas")))
print("(economía OR mercado) AND-NOT (financiero AND pérdidas): ", idx.showDocuments(result))




/tmp/ipykernel_15184/2263868634.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


sostenibilidad AND ambiente AND renovable:  []
tecnología AND (banca OR finanzas):  []
economía AND-NOT inflación:  []
(renovable OR eléctrico) AND energía AND-NOT contaminación:  [5]
(economía OR mercado) AND-NOT (financiero AND pérdidas):  [2, 3, 4, 5]


## 3. (8 puntos) Similitud de Coseno usando el indice invertido
Implementar búsqueda por similitud de coseno aprovechando el índice invertido:

- Proceso de búsqueda:
    - Recibe una consulta en lenguaje natural y un parámetro top_k
    - Utiliza el índice invertido para identificar documentos candidatos
    - Calcula similitud de coseno solo con los documentos relevantes utilizando los pesos TF-IDF
    - Retorna los top-k documentos más similares

<img src="https://1drv.ms/i/c/0c2923df9f1f816f/IQSELMi5qcbqS7lsy5sn8ZLpAZ3G2ciXdabecVJ0vhKoL78" width="500" align="" />

####  Pruebas funcionales

In [33]:
test_queries = [
    "¿Cuáles son las últimas innovaciones en la banca digital y la tecnología financiera?",
    "evolución de la inflación y el crecimiento de la economía en los últimos años",
    "avances sobre sostenibilidad y energías renovables para el medio ambiente"
]

for test in test_queries:    
    results = idx.cosine_search(test['query'], test['top_k'])
    print(f"Top {test['top_k']} documentos más similares:")    
    for doc_id, score in results:
        print(f"Doc {doc_id}: {score:.3f}: ", idx.showDocument(doc_id))

TypeError: string indices must be integers, not 'str'